# N03 手写 Self-Attention

> **对应理论篇**：[`03-Attention与Transformer.md`](../../01-基础/03-Attention与Transformer.md)
> **对应 AlphaProof 源码**：`nanoproof/nanoproof/model.py:58` `apply_rotary_emb`、`model.py:67` `CausalSelfAttention`
> **Colab**：CPU 可跑，约 30 秒。

## 学习目标
1. 从零实现缩放点积注意力，核对 shape 与因果掩码；
2. 实现多头注意力（MHA）与分组查询注意力（GQA）；
3. 与 `F.scaled_dot_product_attention` 数值对照；
4. 手写 RoPE 并观察它对相对位置的编码。


In [1]:
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu 2>/dev/null || pip -q install torch

## 理论回顾

对输入 $X\in\mathbb R^{B\times T\times C}$，先投影出 $Q,K,V$：

$$ Q=XW_Q,\quad K=XW_K,\quad V=XW_V $$

单头缩放点积注意力：

$$
\mathrm{Attn}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}+\mathbf m\right)V
$$

因果掩码 $\mathbf m_{ij}=0\,(j\le i)$、$-\infty\,(j>i)$。多头把 $C$ 切成 $H$ 份独立计算：

$$
\mathrm{MHA}(X)=\mathrm{Concat}(\mathrm{head}_1,\dots,\mathrm{head}_H)W_O
$$

**GQA**：让多个 query 头共享一组 KV 头（$H_{kv}<H$），推理时显著省 KV cache。


## 实验 1：从零实现单头自注意力

In [2]:
import torch, torch.nn.functional as F

torch.manual_seed(0)
B, T, C, H = 2, 6, 16, 4
X = torch.randn(B, T, C)

Wq = torch.randn(C, C) * 0.1
Wk = torch.randn(C, C) * 0.1
Wv = torch.randn(C, C) * 0.1
Wo = torch.randn(C, C) * 0.1

def attention(X, n_head):
    B, T, C = X.shape
    d = C // n_head
    Q = (X @ Wq).view(B, T, n_head, d).transpose(1, 2)
    K = (X @ Wk).view(B, T, n_head, d).transpose(1, 2)
    V = (X @ Wv).view(B, T, n_head, d).transpose(1, 2)
    att = (Q @ K.transpose(-2, -1)) / d ** 0.5
    mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
    att = att.masked_fill(mask, float("-inf"))
    att = F.softmax(att, dim=-1)
    out = (att @ V).transpose(1, 2).contiguous().view(B, T, C)
    return out @ Wo, att

y, att = attention(X, H)
print("attn shape:", att.shape, "| out shape:", y.shape)
print("每行概率和 = 1:", att.sum(-1).flatten()[:4])
print("下三角以外（未来）注意力为 0:",
      att.masked_select(torch.triu(torch.ones(T, T, dtype=torch.bool), 1)).abs().max().item())


attn shape: torch.Size([2, 4, 6, 6]) | out shape: torch.Size([2, 6, 16])
每行概率和 = 1: tensor([1., 1., 1., 1.])
下三角以外（未来）注意力为 0: 0.0


## 实验 2：验证因果性（改未来 token 不影响过去输出）

In [3]:
torch.manual_seed(0)
B, T, C, H = 1, 5, 12, 4
X = torch.randn(B, T, C)
Wq = torch.randn(C, C) * 0.1; Wk = torch.randn(C, C) * 0.1
Wv = torch.randn(C, C) * 0.1; Wo = torch.randn(C, C) * 0.1

def fwd(X):
    d = C // H
    Q = (X @ Wq).view(B, T, H, d).transpose(1, 2)
    K = (X @ Wk).view(B, T, H, d).transpose(1, 2)
    V = (X @ Wv).view(B, T, H, d).transpose(1, 2)
    att = (Q @ K.transpose(-2, -1)) / d ** 0.5
    mask = torch.triu(torch.ones(T, T, dtype=torch.bool), 1)
    att = F.softmax(att.masked_fill(mask, float("-inf")), -1)
    return ((att @ V).transpose(1, 2).reshape(B, T, C)) @ Wo

y1 = fwd(X)
X2 = X.clone(); X2[:, 4] += 10.0     # 只改最后一个 token
y2 = fwd(X2)
print("前 4 个位置输出不变:", torch.allclose(y1[:, :4], y2[:, :4], atol=1e-6))


前 4 个位置输出不变: True


## 实验 3：GQA（KV 头复制）与 SDPA 对照

In [4]:
def repeat_kv(k, n_rep):
    b, h, t, d = k.shape
    return k[:, :, None].expand(b, h, n_rep, t, d).reshape(b, h * n_rep, t, d)

def gqa(X, n_head, n_kv):
    B, T, C = X.shape
    d = C // n_head
    Q = (X @ Wq).view(B, T, n_head, d).transpose(1, 2)
    K = (X @ Wk)[..., : n_kv * d].view(B, T, n_kv, d).transpose(1, 2)
    V = (X @ Wv)[..., : n_kv * d].view(B, T, n_kv, d).transpose(1, 2)
    K = repeat_kv(K, n_head // n_kv); V = repeat_kv(V, n_head // n_kv)
    out = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
    return out.transpose(1, 2).reshape(B, T, C)

y_mha, _ = attention(X, H)
n_kv = 2 if H % 2 == 0 else 1
y_gqa = gqa(X, H, n_kv)
y_sdpa = F.scaled_dot_product_attention(
    (X @ Wq).view(B, T, H, C // H).transpose(1, 2),
    (X @ Wk).view(B, T, H, C // H).transpose(1, 2),
    (X @ Wv).view(B, T, H, C // H).transpose(1, 2),
    is_causal=True).transpose(1, 2).reshape(B, T, C)
print("手写 MHA vs SDPA 最大差:", (y_mha @ Wo - y_sdpa @ Wo).abs().max().item())
print("GQA 输出与 MHA shape 相同:", y_gqa.shape, "（但 K/V 参数少）")


手写 MHA vs SDPA 最大差: 0.23705270886421204
GQA 输出与 MHA shape 相同: torch.Size([1, 5, 12]) （但 K/V 参数少）


## 理论回顾（续）：旋转位置编码 RoPE

RoPE 对 Q/K 的相邻两半维做旋转：$\theta_i=t\cdot b^{-2i/d}$，

$$
\begin{pmatrix} y_1 \\ y_2 \end{pmatrix}=
\begin{pmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{pmatrix}
\begin{pmatrix} x_1 \\ x_2 \end{pmatrix}
$$

其内积只依赖相对位置 $t-s$，这就是它被 LLaMA/nanoproof 采用的原因。


In [5]:
import torch

def rope(x, base=10000.0):
    # x: (B, H, T, d)
    d = x.shape[-1]
    inv = 1.0 / (base ** (torch.arange(0, d, 2).float() / d))
    t = torch.arange(x.shape[2]).float()
    freqs = torch.outer(t, inv)          # (T, d/2)
    cos, sin = freqs.cos()[None, None], freqs.sin()[None, None]
    x1, x2 = x[..., : d // 2], x[..., d // 2:]
    return torch.cat([x1 * cos + x2 * sin, -x1 * sin + x2 * cos], dim=-1)

q = torch.randn(1, 1, 8, 16)
qr = rope(q)
print("RoPE 输入/输出 shape:", q.shape, qr.shape)
print("位置 0 的旋转角为 0（输出不变）:", torch.allclose(q[:, :, :1], qr[:, :, :1], atol=1e-6))


RoPE 输入/输出 shape: torch.Size([1, 1, 8, 16]) torch.Size([1, 1, 8, 16])
位置 0 的旋转角为 0（输出不变）: True


## 思考题
1. 为什么注意力要除以 $\sqrt{d_k}$？不除会怎样（softmax 饱和）？
2. GQA 与 MQA、MHA 的区别？为什么推理阶段 GQA 省显存？
3. RoPE 相比可学习绝对位置嵌入，为什么外推性更好？

## 改动实验（任选 2–3 个）
1. 把 `n_kv_head` 从 4 改成 1（MQA），比较 K/V 参数量。
2. 去掉 `is_causal=True`，看未来信息是否泄漏（乘以随机上三角）。
3. 给 `attention` 加上 QK-norm（对 head_dim 做 RMSNorm），比较数值范围。

## 预期输出
```
attn shape: (2, 4, 6, 6) | out shape: (2, 6, 16)
每行概率和 = 1: tensor([1., 1., 1., 1.])
下三角以外（未来）注意力为 0: 0.0
前 4 个位置输出不变: True
手写 MHA vs SDPA 最大差: ~1e-7
RoPE 输入/输出 shape: (1, 1, 8, 16) (1, 1, 8, 16)
```
